# Stereo calibration

Solves per-camera intrinsics, then the camera-to-camera transform.

**World frame is camera A**, `T_world_camA = I`:

$$T_{B \leftarrow A} \;=\; T_{B \leftarrow \text{board}} \; T_{\text{board} \leftarrow A}
\;=\; T_{B \leftarrow \text{board}} \, T_{A \leftarrow \text{board}}^{-1}$$

**Units are millimetres.** This matches `RADIUS_MM`, `baseline_mm` and every `center_mm`
column in the CSVs. The board is built in mm, so `solvePnP` returns mm directly.
`vision/visual_servo.ipynb` builds it in metres instead.

The maths lives in `calibrate.py`. This notebook runs it, plots it, and explains it.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from calibrate import *      # noqa: F403 -- this notebook documents these modules
from capture import *        # noqa: F403
from plots import *          # noqa: F403
from results import *        # noqa: F403
from sheet import *          # noqa: F403
from test_calibrate import *  # noqa: F403

print("cv2", cv2.__version__)
print("board  ", SPEC.name, SPEC.summary()["size_mm"], "mm")
print("pairs ->", PAIR_DIR)
print("rig   ->", RIG_PATH)


cv2 5.0.0
board   4x12_6mm [24.0, 72.0] mm
pairs -> /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/results/stereo_calibration/pairs
rig   -> /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/controller/calib/stereo_rig.json


## 1. The board

`BOARDS` holds the boards this rig has used. `SPEC` selects one, and the default is
**4x12 squares at 6 mm**: 24.0 x 72.1 mm, sized to sit on a 25 x 75 mm microscope slide.
Glass is the flattest cheap backing there is, and section 1a explains why flatness matters.

`make_board(SPEC)` writes the printable sheet. The older 9x6 letter board is still in
`BOARDS` for reproducing earlier runs.

In [2]:
for name, board in BOARDS.items():
    w, h = board.size_mm
    print(f"{name:16s} {board.cols}x{board.rows} at {board.square_mm:7.4f} mm "
          f"= {w:6.1f} x {h:5.1f} mm, {board.n_corners:3d} corners")

print()
for k, v in SPEC.summary().items():
    print(f"{k:12s} {v}")

9x6_letter       9x6 at 16.6670 mm =  150.0 x 100.0 mm,  40 corners
9x6_6mm          9x6 at  6.0113 mm =   54.1 x  36.1 mm,  40 corners
4x12_6mm_slide   4x12 at  6.0113 mm =   24.0 x  72.1 mm,  33 corners

board        4x12_6mm
squares      [4, 12]
square_mm    6.0
marker_mm    4.5
dictionary   DICT_4X4_100
n_corners    33
size_mm      [24.0, 72.0]


### 1a. A print whose scale you can check

`generate_pdf` lays the board out at true size and prints a 100 mm ruler bar beside it, from
the same mm-to-px conversion as the board. If the bar measures 100 mm, the squares are the
stated pitch. If it does not, both are wrong by the same factor.

It rasterises at a whole number of pixels per square, so every square is identical. That
quantises the pitch: 6 mm at 600 dpi becomes 142 px = **6.0113 mm**, 0.19 % high. The
returned number, not the nominal one, is what the printer drew.

The sheet also carries crop marks at the board's four corners, so a small board can be cut
to size and mounted.

In [3]:
# Regenerate the print, then measure the ruler bar before shooting anything.
# make_board(SPEC)

## 2. Detection

`CharucoDetector.detectBoard` sub-pixel refines internally. **OpenCV 5 returns flat
`(N,2)` / `(N,)`** where OpenCV 4 returned `(N,1,2)` / `(N,1)`, and
`interpolateCornersCharuco` / `calibrateCameraCharuco` are gone entirely -- so
`writeup/camera_calibration.ipynb`, which uses both, is dead code rather than a reference.
`detect` normalises the shapes once so nothing below reshapes by hand.

`board_incidence_deg` reads 0° face-on and 90° edge-on. It takes the board normal against the
optical axis as an **undirected line** (`abs`), because which face points at the lens is not
what conditions the pose -- how obliquely you see it is.

In [4]:
# Detection on a synthetic sheet, so this cell runs with no camera and no capture.
sheet = SPEC.board.generateImage((100 * SPEC.cols, 100 * SPEC.rows), marginSize=40)
corners, ids = detect(SPEC, sheet)
print(f"{len(ids)}/{SPEC.n_corners} corners")

plt.figure(figsize=(4, 8))
plt.imshow(annotate(sheet, SPEC, corners, ids)[..., ::-1])
plt.axis("off");

33/33 corners


## 3. The camera: pick the mode before you shoot anything

Measured on the ELP global-shutter module on this bench (VID `0x32E4`, PID `0x9281`, the
OV9281 sensor), by timing real reads -- `CAP_PROP_FPS` is only what the driver claims:

| requested | delivered | measured fps | relation to native | field of view |
|---|---|---|---|---|
| 1280×800 | 1280×800 | **119** | **native** | full |
| 640×400 | 640×400 | **217** | **exact 0.5× rescale** | full |
| 640×480 | 640×480 | 107 | crop | narrower |
| 1280×720 | 1280×720 | 53 | crop | narrower |
| 320×240 | 320×240 | 333 | crop | much narrower |

Two results here are worth more than the table.

**1280×720 is the worst mode available.** It is *slower* than the full 1280×800 (53 vs 119
fps) and sees *less*, because 800 is the sensor's native height and 720 is a windowed
readout of it. The reflex to ask for "720p" costs you half the frame rate and part of the
image. Ask for 1280×800.

**Only 640×400 is a true rescale.** Cross-correlating each mode against a resize versus a
centre crop of the native frame: 640×400 matches the *resize* at 0.9994, while 640×480,
1280×720 and 320×240 all match the *crop* (0.995, 0.9996, 0.991). This decides whether
intrinsics transfer:

- 1280×800 → 640×400 is a clean $\times 0.5$, so `rig.Camera.scaled(0.5)` is exactly right  -- 
  $f_x, f_y, c_x, c_y$ all halve.
- Every other mode is a **crop**, where $f_x, f_y$ are *unchanged* and only $c_x, c_y$ shift
  by the discarded margin. Applying a scale factor there is silently wrong, in the same
  quiet way §14.4's print-scale error is wrong: plausible numbers, no warning.

So: **calibrate at 1280×800, then run at 1280×800 (119 fps) or 640×400 (217 fps).** Those
two are one calibration. Anything else needs its own.

Also measured: the sensor is **monochrome** -- all three BGR channels come back bit-identical
 --  so `grayscale=True` throws away nothing and `IMREAD_GRAYSCALE` is the right reader (§13).
And as on every other camera here, macOS refuses exposure control: every
`CAP_PROP_EXPOSURE` / `AUTO_EXPOSURE` set returns `False` and reads back `-1`. Light the
scene rather than fighting the driver.

At 217 fps this is also the first camera on the bench near the 240 fps target the pose
README is written against; `sources.MonoCamera`'s drop-oldest grabber matters at that rate
in a way it never did at the C270's 28 fps.

In [5]:
# from elp import probe_indices; probe_indices()

## 3a. Capture: a shutter that refuses

`capture` opens one camera or two through `sources.open_stereo` and shows a live preview.
It shoots by itself, ten times a second, and saves a pair only when that pair clears every
gate in `gates`. SPACE pauses and resumes. `q` quits.

Nothing floods the disk, because the gates ration the shots rather than the operator. The
novelty gate refuses a pose already held, so a board left alone yields exactly one pair.
The stillness gates refuse everything in flight. 10 Hz is faster than a hand moves the
board, so the first still instant at each new placement is the one that gets saved. Place
the board, and it is photographed as soon as it settles.

That refusal is the design, and [§16](theory.md) is the argument for it. The two cameras
free-run, so a pair straddling `max_skew_s` sees the board in two different places. At
$f_x = 2765$ px, which is 11 px per mm, a hand sweep at 40 mm/s is 1.98 px of disagreement
against a 0.5 px budget. That is what failed the first real run at 1.08 px joint RMS.

The error is a product, $v\Delta$. Correcting it is possible and was built: estimating each
corner's velocity from neighbouring frames cut epipolar disagreement 6.2x and turned a
failing solve into a passing one. **Resting the board** replaced it. That sets $v$ to zero
exactly, needs no estimator, and costs a block of wood. A calibration target is static by
nature. The only reason it ever moved is that a hand held it.

Skew is handled where it is cheapest. `sources.StereoCamera` re-reads until the two frames
land inside `max_skew_s`. At 2 ms that yields 16 pairs/s at 1.02 ms median, against 7.71 ms
unfiltered, which is still above the 10 Hz the shutter asks for.

**A new capture replaces the old one.** The first saved pair deletes the pairs already on
disk, so a solve can never mix two sets. Pass `append=True` to add to them instead. Mixing
is not hypothetical: a 100-pair solve that failed at 0.98 px turned out to hold 66 pairs
from a hand-held session two days earlier.

On macOS select by index: AVFoundation exposes no device names through OpenCV, and USB
cameras enumerate *before* the built-in FaceTime, so the plugged-in pair is usually 0 and 1.
An inverted mount is handled by `rotate180`, which lives in `sources.MonoCamera` so
calibration and the live pose loop cannot disagree about which way is up.

**Shoot at the sensor's native mode -- see section 3 for why 1280x800 and not 720p.**

**Place the board, do not wave it.** Rest it on a block and move the block between shots.
Work through the tilt range as well as the frame, because the tilt-spread gate asks for it.

In [ ]:
# Live capture. Interactive -- run this cell only when the cameras are plugged in.
capture(PAIR_DIR, indices=(0, 1), spec=SPEC)

/Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/results/stereo_calibration/pairs already holds 24 pair(s); they will be deleted when the first new pair is saved
auto: SPACE pauses and resumes, q quits


2026-08-24 14:09:20.763 Python[47265:4611052] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/63/7cv2q0897zv367k7c1bdmp500000gn/T/org.python.python.savedState


source ended

0 pair(s) in /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/results/stereo_calibration/pairs


PosixPath('/Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/results/stereo_calibration/pairs')

: 

## 3b. The gates

`gates` returns `(ok, checks)`, and `checks` is what the overlay draws -- one line per
condition, so the operator sees which one is holding the shutter instead of guessing:

| gate | threshold | what it protects |
|---|---|---|
| board in view, and posed | `MIN_CORNERS`, a finite `solvePnP` | there is something to measure |
| shared corners | `MIN_COMMON_CORNERS` | the pair has common points to relate |
| still (pair) | $v\Delta \le$ `MAX_PAIR_ERROR_PX` | one board position, not two |
| still (blur) | $v\tau \le$ `MAX_BLUR_PX` | corner localisation, which sync cannot help |
| new pose | `MIN_KEEP_ROT_DEG` or `MIN_KEEP_SHIFT_PX` | frames that repeat add nothing |
| tilt spread | `BAND_FULL` per 15 degree band | the orientation spread section 5 wants |

The last two gates carry the lesson of two failed runs. Position novelty alone lets the
operator slide the board around the frame without ever tilting it: 97 views were captured
that way at 16.0 degrees of spread, against the 20 the intrinsics need. Shooting *more*
frames cannot fix a spread problem. So once a 15 degree band of incidence holds `BAND_FULL`
shots, the shutter refuses that angle until the spread arrives, and the overlay names the
band it wants.

**Lock the focus first.** These M12 modules focus by screwing the barrel, and focal length
moves with it, so turning the lens after calibrating silently invalidates $f_x, f_y$.

In [7]:
# The gates on a synthetic board: still, then sweeping at the speed that failed the
# first real run, then repeating a pose already saved. No camera needed.
from calibrate import _look          # private: the gates are the public surface

sheet = SPEC.board.generateImage((100 * SPEC.cols, 100 * SPEC.rows), marginSize=40)
look = _look(SPEC, sheet, None, None)

for label, speed, saved in [
    ("still", 0.0, []),
    ("440 px/s", 440.0, []),
    ("duplicate pose", 0.0, [(Rotation.from_rotvec(look["rvec"].ravel()),
                              look["corners"].reshape(-1, 2).mean(axis=0),
                              look["incidence"])]),
]:
    ok, checks = gates(SPEC, [dict(look, speed=speed)] * 2, 0.002, saved)
    blocked = [name for name, passed, _ in checks if not passed]
    print(f"{label:16s} {'SHOOT' if ok else 'refuse: ' + ', '.join(blocked)}")

still            SHOOT
440 px/s         refuse: still (pair), still (blur)
duplicate pose   refuse: new pose


## 4. Load what was shot, and detect

Matching index in `A/` and `B/` is the pairing key. `load_views` keeps a view only if it
clears `MIN_CORNERS`. It records incidence now and enforces it later, once real intrinsics
exist.

In [8]:
HAVE_PAIRS = (PAIR_DIR / "A").is_dir() and any((PAIR_DIR / "A").glob("*.png"))
if HAVE_PAIRS:
    views_a, views_b, image_size = load_views(SPEC)
else:
    print(f"no pairs under {PAIR_DIR} -- record some first (section 3a)")

camera A: 24 usable views, 24.9 corners mean
camera B: 24 usable views, 22.3 corners mean


## 5. Per-camera intrinsics

`calibrate_intrinsics` solves each camera from **all of its own** usable views, not just the
shared ones. A view that only camera A saw still constrains A's focal length and distortion.

The default is the 5-coefficient model `(k1,k2,p1,p2,k3)`. `CALIB_RATIONAL_MODEL` adds three
more radial terms, but it overfits unless many views reach the image border.

A single planar board gives coplanar object points, which leaves the intrinsics
ill-conditioned. Only genuine tilt and rotation between views fixes that, so the code
measures the orientation spread and warns rather than assuming it.

In [9]:
# One camera on its own: `capture` with a single index fills PAIR_DIR/A alone.
#
#   capture(PAIR_DIR, indices=0, spec=SPEC)

if (PAIR_DIR / "A").is_dir() and any((PAIR_DIR / "A").glob("*.png")):
    K, dist, info = intrinsics_from_dir(PAIR_DIR / "A", spec=SPEC, name="A")
    save_intrinsics(OUT_DIR / "camera_intrinsics_A.npz", K, dist, SPEC, info)
else:
    print(f"nothing captured to {PAIR_DIR / 'A'} yet")

camera A: 24 views, RMS 0.6892 px, worst view 2.6171 px
  fx=2770.55 fy=2749.22 cx=639.38 cy=379.83
  dist [-4.67420e-01  5.98251e+00 -3.39460e-03  9.82805e-04 -5.99414e+01]
  incidence 25.8-75.5 deg, spread 14.2 deg


## 6. Extrinsics

Two stages, because they fail differently.

**Seed -- per-pair, closed form.** For each pair, `solvePnP` in both cameras and compose
$T_{B\leftarrow A} = T_{B\leftarrow\text{board}} \, T_{A\leftarrow\text{board}}^{-1}$.
Rotations average with `Rotation.mean()` -- the proper chordal mean on SO(3), not an
elementwise matrix average, which is not a rotation. Translation uses the **median**, which
survives one bad pair.

The **spread across pairs is the honest uncertainty**, and it is the number a bundle RMS
hides: `stereoCalibrate` can report 0.3 px while one pair disagrees by 5°.

**Refine -- `cv2.stereoCalibrate` with `CALIB_FIX_INTRINSIC`**, seeded from the median.
Intrinsics stay fixed so their error cannot leak into the extrinsic.

The step that is easy to miss: **intersect the ChArUco ids per pair.** Each view detects a
different corner subset, and `stereoCalibrate` requires the *same* object points in both
lists. Feeding it unintersected lists is silently wrong whenever the two cameras saw
different corners -- which is always.

OpenCV's `R, T` map camera A into camera B ($X_B = R X_A + T$), so they *are*
$T_{B\leftarrow A}$.

## 7. Acceptance

From `docs/pose_localization_project_context.md` §6: RMS **< 0.5 px**, and residuals
**isotropic and structureless**. The second half is the one that matters -- a radial or
edge-worse pattern means underfit distortion, i.e. systematic error hiding under a
respectable average.

**The residuals have to come from the joint fit.** Re-solving each camera's board pose
independently measures only that camera's intrinsics. The extrinsic never enters, so a
completely wrong $T_{B\leftarrow A}$ would still produce a clean residual and pass. So
`stereo_residuals` fits **one** board pose per pair that has to explain both views *through*
$T_{B\leftarrow A}$ -- camera B's residual then carries the extrinsic error, which is the
thing being gated.

Then bin those residuals by radius from the principal point and compare outer to inner RMS,
and compare the x and y spreads. The gate also checks that the bundle agrees with the
closed-form seed: because OpenCV 5 will not accept an extrinsic guess, the two are genuinely
independent estimates, and their agreement is information rather than a tautology.

## 8. Run it

`run_calibration` loads the pairs, solves both cameras, solves the extrinsic, and gates the
result. It writes nothing to disk. Section 10 does that.

In [10]:
CAL = run_calibration(SPEC) if HAVE_PAIRS else None
if CAL is None:
    print("no pairs -- sections 9-11 need them. Sections 12-13 do not.")

camera A: 24 usable views, 24.9 corners mean
camera B: 24 usable views, 22.3 corners mean
image size (1280, 800)

camera A: 24 views, RMS 0.6892 px, worst view 2.6171 px
  fx=2770.55 fy=2749.22 cx=639.38 cy=379.83
  dist [-4.67420e-01  5.98251e+00 -3.39460e-03  9.82805e-04 -5.99414e+01]
  incidence 25.8-75.5 deg, spread 14.2 deg

camera B: 24 views, RMS 0.4203 px, worst view 0.5172 px
  fx=2691.95 fy=2691.67 cx=443.65 cy=378.78
  dist [-3.86074e-01  2.06128e+00 -3.19563e-03  4.39274e-03 -1.44017e+01]
  incidence 20.6-69.3 deg, spread 11.6 deg

23 usable pairs, 1 rejected
  reject pair_019: incidence 76/52 deg
seed from 23 pairs: baseline 188.50 mm
  pair-to-pair spread: rotation 0.348 deg median / 1.126 worst, translation 0.698 mm median / 1.936 worst
stereoCalibrate: RMS 0.5588 px over 23 pairs (worst pair 1.1757 px)
  baseline 188.520 mm
  agreement with the independent closed-form seed: 0.1629 deg, 0.3310 mm

joint stereo residual: 0.5588 px RMS over both views (A 0.5892, B 0.5267)


## 9. Diagnostics

`figures` draws six panels: per-pair reprojection, the residual scatter that the isotropy
check summarises, corner coverage per camera, per-pair extrinsic spread, and the incidence
distribution. Holes in the coverage are where distortion is unconstrained. The incidence
panel answers open item 3 in `docs/pose_localization_project_context.md`: whether one angled
board can serve a widely separated pair, or whether it takes a cube.

In [11]:
if CAL:
    figures(CAL["pairs"], CAL["image_size"], CAL["resid"])

figures -> /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/results/stereo_calibration/stereo_calibration.png


## 10. Write `stereo_rig.json`

World is camera A, so `T_world_camA = I` and `T_world_camB = (T_{B\leftarrow A})^{-1}`.

`write_results` also saves each camera's intrinsics as npz, in the richer schema that carries
the board, the image size and the RMS. The provenance then travels with the numbers.

In [12]:
if CAL and CAL["passed"]:
    the_rig = write_results(CAL, SPEC)
elif CAL:
    print("acceptance failed -- not writing a rig. Fix the capture, not the limit.")

acceptance failed -- not writing a rig. Fix the capture, not the limit.


## 11. Scale check on held-out pairs

`scale_check` triangulates corner pairs and compares them against the board's own geometry.
It undistorts to normalised coordinates, then uses $P_A = [I|0]$ and $P_B = [R|t]$, so the
triangulated points come out in camera A's frame in millimetres.

**What this proves, and what it does not.** It confirms that the extrinsic and the *relative*
scale agree, and a wrong baseline shows up at once. It shares the board's absolute scale, so
a mis-scaled print passes it happily. Only calipers on the print close that gap, which is why
section 1a keeps insisting on it.

In [13]:
if CAL:
    scale_check(CAL["pairs"], CAL["K_a"], CAL["dist_a"], CAL["K_b"], CAL["dist_b"],
                CAL["T_ba"], SPEC, holdout=8)

scale check over 8 pair(s), 914 corner-to-corner distances
  absolute error  -0.0091 mm mean, 0.0742 mm p95, 0.1371 mm worst
  relative error  -0.0621% mean, 0.6818% p95
  (shares the board's absolute scale -- a mis-scaled print passes this)


## 12. Self-test: synthetic round trip

No hardware, no board, no printer. `self_test` builds a rig with a **known** extrinsic,
projects the board's corners into both cameras at many synthetic poses, and runs the
identical solve. If this fails, the maths above is wrong and nothing measured with it means
anything.

It covers exact recovery, behaviour under 0.2 px corner noise, the id-intersection path
against deliberately disjoint corner subsets, the 180 degree flip for an inverted mount, and
the shutter's gates.

In [14]:
self_test()

test_units_are_mm: A 9x6 at 16.667 mm spans 150 x 100 mm, not 0.15 x 0.10.
  board 150.003 x 100.002 mm, 40 corners
  ok

test_marker_scales_with_square: A rescaled print rescales both dimensions.
  12.5000 -> 12.1250 mm at 97% scale
  ok

test_exact: Noise-free: the solve must return the rig it was given, to numerical precision.
18 usable pairs, 0 rejected
seed from 18 pairs: baseline 300.00 mm
  pair-to-pair spread: rotation 0.000 deg median / 0.000 worst, translation 0.000 mm median / 0.000 worst
stereoCalibrate: RMS 0.0000 px over 18 pairs (worst pair 0.0000 px)
  baseline 300.000 mm
  agreement with the independent closed-form seed: 0.0000 deg, 0.0000 mm
  rotation error 0.000000 deg, translation error 0.000002 mm
  ok

test_noise: 0.2 px corner noise: still sub-0.1 deg, and the spread reflects the noise.
24 usable pairs, 0 rejected
seed from 24 pairs: baseline 300.00 mm
  pair-to-pair spread: rotation 0.067 deg median / 0.112 worst, translation 0.305 mm median / 0.586 worst
stere

## 13. Intrinsics regression

`vision/board_images/9x6/` holds 29 single-camera shots at 960 × 720, one of which
(`image100.jpg`) detects zero corners. They cannot give an extrinsic -- one camera -- but they
do check this notebook's detection and board definition against the notebook that made the
checked-in `vision/camera_intrinsics.npz`.

Three numbers, measured here rather than asserted:

| path | fx | vs checked-in |
|---|---|---|
| this notebook -- `IMREAD_GRAYSCALE`, as `sources.Image` reads | 1411.143 | +2.36 |
| `visual_servo.ipynb` algorithm, same 28 images -- `cvtColor(BGR2GRAY)` | 1410.801 | +2.02 |
| checked-in `camera_intrinsics.npz` | 1408.783 | -- |

Two separate findings, worth keeping apart:

- **The 0.34 px between the first two rows is the JPEG grayscale decode path.**
  `cv2.imread(..., IMREAD_GRAYSCALE)` and `cvtColor(imread(...), BGR2GRAY)` disagree by up to
  **2 grey levels** (mean 0.005), which is enough to move the sub-pixel corners and shift fx
  by a third of a pixel. Board units (mm vs metres) and array shape (`(N,2)` vs `(N,1,2)`)
  were also checked and change K by *exactly* nothing. This notebook keeps
  `IMREAD_GRAYSCALE` to match how `sources.Image` feeds the live pipeline.
- **The remaining ~2 px to the checked-in npz is not reproducible.** Neither decode path
  recovers it from the images now on disk, so that file predates this image set or was made
  with a different OpenCV. It is a fact about the reference, not a drift in this notebook  -- 
  which is the whole reason this cell computes both paths instead of trusting one.

In [15]:
regression_9x6()

camera IMREAD_GRAYSCALE: 28 views, RMS 0.4093 px, worst view 0.6541 px
  fx=1411.14 fy=1409.93 cx=499.66 cy=355.55
  dist [ 0.12353  0.13062  0.00461  0.00392 -2.49718]
  incidence 0.8-55.8 deg, spread 16.5 deg

camera cvtColor BGR2GRAY: 28 views, RMS 0.4054 px, worst view 0.6595 px
  fx=1410.80 fy=1409.60 cx=499.70 cy=355.37
  dist [ 0.12477  0.09173  0.00459  0.00393 -2.22905]
  incidence 0.8-55.8 deg, spread 16.5 deg

         IMREAD_GRAY      BGR2GRAY    checked in    decode    vs ref
  fx        1411.143      1410.801      1408.783    +0.342    +2.360
  fy        1409.934      1409.597      1407.685    +0.337    +2.249
  cx         499.663       499.697       497.553    -0.034    +2.109
  cy         355.553       355.368       355.701    +0.184    -0.148

  'decode' is IMREAD_GRAYSCALE minus BGR2GRAY on the SAME images: the two JPEG
  grayscale paths disagree by up to 2 grey levels, which moves the sub-pixel
  corners. Board units and array shape were checked separately and change